In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


In [6]:
df = pd.read_csv("medicaid-provider-spending.csv")
print(df.columns.tolist())  # column names

/var/folders/dj/z48f7b2j74j0qb_5_slvd9w80000gn/T/ipykernel_81463/2520810276.py:1: DtypeWarning: Columns (0: BILLING_PROVIDER_NPI_NUM) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("medicaid-provider-spending.csv")


['BILLING_PROVIDER_NPI_NUM', 'SERVICING_PROVIDER_NPI_NUM', 'HCPCS_CODE', 'CLAIM_FROM_MONTH', 'TOTAL_UNIQUE_BENEFICIARIES', 'TOTAL_CLAIMS', 'TOTAL_PAID']


In [7]:
# categorize cardiovascular-related HCPCS/CPT codes
# based on the ranges you listed in the conversation

def _cv_category(code):
    if pd.isna(code):
        return None
    s = str(code).strip()
    # C‑codes for assorted devices
    if s.startswith("C"):
        try:
            n = int(s[1:])
        except ValueError:
            return None
        if 2617 <= n <= 2631:
            return "Assorted Cardiovascular and Genitourinary Devices"
        return None
    # numeric codes (most CPT/HCPCS are numeric)
    try:
        n = int(s)
    except ValueError:
        return None

    # defined cardiovascular ranges with descriptive labels
    ranges = [
        (92920, 92998, "Therapeutic Cardiovascular Services and Procedures"),
        (93000, 93153, "Cardiography Procedures"),
        (93150, 93153, "Phrenic Nerve Stimulation System"),
        (93224, 93278, "Cardiovascular Monitoring Services"),
        (93279, 93298, "Implantable, Insertable, and Wearable Cardiac Device Evaluations"),
        (93303, 93356, "Echocardiography Procedures"),
        (93451, 93598, "Cardiac Catheterization Procedures"),
        (93600, 93662, "Intracardiac Electrophysiological Procedures/Studies"),
        (93668, 93668, "Peripheral Arterial Disease Rehabilitation"),
        (93701, 93790, "Non-invasive Physiologic Studies and Procedures"),
        (93792, 93793, "Home and Outpatient International Normalized Ratio (INR) Monitoring Services"),
        (93797, 93799, "Other Cardiovascular Procedures"),
    ]
    for lo, hi, label in ranges:
        if lo <= n <= hi:
            return label
    return None

# apply to the dataframe

df["CARDIOVASCULAR_CATEGORY"] = df["HCPCS_CODE"].apply(_cv_category)
# binary flag for any cardiovascular entry

df["IS_CARDIOVASCULAR"] = df["CARDIOVASCULAR_CATEGORY"].notna().astype("Int8")

# quick sanity check: codes beginning with 92/93 that weren't labeled

unmatched = df["HCPCS_CODE"].loc[
    df["HCPCS_CODE"].astype(str).str.match(r"^(92|93)") & df["IS_CARDIOVASCULAR"].eq(0)
].unique()

print("unmatched codes starting with 92/93:", sorted(unmatched)[:20])
print("total CV rows", df["IS_CARDIOVASCULAR"].sum())

# show distribution by category
print(df.groupby("CARDIOVASCULAR_CATEGORY").size().sort_values(ascending=False).head(10))

unmatched codes starting with 92/93: ['92002', '92004', '92012', '92014', '92015', '92018', '92019', '92020', '92025', '92060', '92065', '92066', '92071', '92072', '92081', '92082', '92083', '92100', '9212', '9213']
total CV rows 4477386
CARDIOVASCULAR_CATEGORY
Cardiography Procedures                                                         3078944
Echocardiography Procedures                                                     1133112
Implantable, Insertable, and Wearable Cardiac Device Evaluations                 152264
Cardiovascular Monitoring Services                                                75584
Cardiac Catheterization Procedures                                                14500
Home and Outpatient International Normalized Ratio (INR) Monitoring Services       9724
Other Cardiovascular Procedures                                                    5525
Non-invasive Physiologic Studies and Procedures                                    3841
Assorted Cardiovascular and Genito

In [8]:
# categorize neurology-related HCPCS/CPT codes
# ranges sourced from Codes.xlsx (Neuro sheet)

def _neuro_category(code):
    if pd.isna(code):
        return None
    s = str(code).strip()

    # numeric codes (most CPT/HCPCS are numeric)
    try:
        n = int(s)
    except ValueError:
        return None

    ranges = [
        (95700, 95811, "Sleep Medicine Testing and Long-term EEG Procedures"),
        (95812, 95830, "Routine Electroencephalography (EEG) Procedures"),
        (95829, 95836, "Electrocorticography"),
        (95836, 95857, "Range of Motion Testing"),
        (95860, 95872, "Electromyography Procedures"),
        (95873, 95887, "Ischemic Muscle Testing and Chemodenervation Guidance Procedures"),
        (95905, 95913, "Nerve Conduction Tests"),
        (95919, 95924, "Autonomic Function Testing Procedures"),
        (95925, 95941, "Evoked Potentials and Reflex Testing Procedures"),
    ]

    for lo, hi, label in ranges:
        if lo <= n <= hi:
            return label
    return None

# apply to the dataframe
df["NEUROLOGY_CATEGORY"] = df["HCPCS_CODE"].apply(_neuro_category)
# binary flag for any neurology entry
df["IS_NEUROLOGY"] = df["NEUROLOGY_CATEGORY"].notna().astype("Int8")

# quick sanity check: codes beginning with 95/96 that were not labeled
unmatched_neuro = df["HCPCS_CODE"].loc[
    df["HCPCS_CODE"].astype(str).str.match(r"^(95|96)") & df["IS_NEUROLOGY"].eq(0)
].unique()

print("unmatched codes starting with 95/96:", sorted(unmatched_neuro)[:20])
print("total neuro rows", df["IS_NEUROLOGY"].sum())

# show distribution by category
print(df.groupby("NEUROLOGY_CATEGORY").size().sort_values(ascending=False).head(10))


unmatched codes starting with 95/96: ['95004', '95012', '95017', '95018', '95024', '95027', '95028', '9504', '95044', '95060', '95065', '95070', '95076', '95079', '95115', '95117', '95120', '95125', '95144', '95145']
total neuro rows 390312
NEUROLOGY_CATEGORY
Sleep Medicine Testing and Long-term EEG Procedures                 146037
Ischemic Muscle Testing and Chemodenervation Guidance Procedures     83858
Routine Electroencephalography (EEG) Procedures                      75576
Nerve Conduction Tests                                               44233
Autonomic Function Testing Procedures                                19435
Evoked Potentials and Reflex Testing Procedures                      17488
Electromyography Procedures                                           2384
Range of Motion Testing                                                844
Electrocorticography                                                   457
dtype: int64


In [9]:
df["CLAIM_FROM_MONTH"] = pd.to_datetime(df["CLAIM_FROM_MONTH"])

# keep cardiovascular or neurology codes only
cardio = df[df["IS_CARDIOVASCULAR"] == 1 ].copy()
neuro = df[df["IS_NEUROLOGY"] == 1 ].copy()

# drop missing codes if needed
cardio = cardio.dropna(subset=["HCPCS_CODE"])
neuro = neuro.dropna(subset=["HCPCS_CODE"])

combined = pd.concat([cardio, neuro], ignore_index=True)
combined.sample(20)

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CARDIOVASCULAR_CATEGORY,IS_CARDIOVASCULAR,NEUROLOGY_CATEGORY,IS_NEUROLOGY
2290334,1992779482,1992779482,93005,2021-06-01,60,78,258.52,Cardiography Procedures,1,NaN,0
4324494,1356766513,1417956707,93000,2019-07-01,28,29,0.00,Cardiography Procedures,1,NaN,0
1527916,1548210198,1992714166,93306,2018-03-01,21,26,528.05,Echocardiography Procedures,1,NaN,0
425176,1740434851,1740434851,93306,2018-10-01,32,37,2329.99,Echocardiography Procedures,1,NaN,0
1365806,1982653515,1730139502,93306,2019-12-01,14,14,623.30,Echocardiography Procedures,1,NaN,0
2148683,1598835308,1063494581,93005,2020-11-01,12,12,292.50,Cardiography Procedures,1,NaN,0
2662078,1336139799,1366432718,93018,2020-10-01,21,22,190.06,Cardiography Procedures,1,NaN,0
828480,1992975775,1144458027,93306,2018-07-01,20,20,1168.71,Echocardiography Procedures,1,NaN,0
2364910,1013950807,1891745501,93000,2022-08-01,13,13,242.78,Cardiography Procedures,1,NaN,0
3914985,1285686303,1578852513,93010,2024-01-01,12,12,51.94,Cardiography Procedures,1,NaN,0


In [10]:
combined["CATEGORY_DESCRIPTION"] = (
    combined["CARDIOVASCULAR_CATEGORY"]
    .fillna(combined["NEUROLOGY_CATEGORY"])
)
combined["SPECIALTY"] = np.select(
    [
        combined["IS_CARDIOVASCULAR"] == 1,
        combined["IS_NEUROLOGY"] == 1
    ],
    [
        "Cardiovascular",
        "Neurology"
    ],
    default="Other"
)

combined

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CARDIOVASCULAR_CATEGORY,IS_CARDIOVASCULAR,NEUROLOGY_CATEGORY,IS_NEUROLOGY,CATEGORY_DESCRIPTION,SPECIALTY
0,1790869402,1790869402,93229,2024-04-01,1414,1429,750233.49,Cardiovascular Monitoring Services,1,NaN,0,Cardiovascular Monitoring Services,Cardiovascular
1,1790869402,1790869402,93229,2024-05-01,1380,1414,741462.47,Cardiovascular Monitoring Services,1,NaN,0,Cardiovascular Monitoring Services,Cardiovascular
2,1790869402,1790869402,93229,2024-03-01,1407,1452,725990.50,Cardiovascular Monitoring Services,1,NaN,0,Cardiovascular Monitoring Services,Cardiovascular
3,1790869402,1790869402,93229,2022-08-01,1298,1340,685090.30,Cardiovascular Monitoring Services,1,NaN,0,Cardiovascular Monitoring Services,Cardiovascular
4,1790869402,1790869402,93229,2024-01-01,1315,1392,667647.93,Cardiovascular Monitoring Services,1,NaN,0,Cardiovascular Monitoring Services,Cardiovascular
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4867693,1407043466,1407043466,95921,2021-06-01,23,23,-1731.43,NaN,0,Autonomic Function Testing Procedures,1,Autonomic Function Testing Procedures,Neurology
4867694,1487652053,1487652053,95800,2018-09-01,18,24,-1793.23,NaN,0,Sleep Medicine Testing and Long-term EEG Proce...,1,Sleep Medicine Testing and Long-term EEG Proce...,Neurology
4867695,1487652053,1487652053,95800,2018-04-01,22,30,-1951.86,NaN,0,Sleep Medicine Testing and Long-term EEG Proce...,1,Sleep Medicine Testing and Long-term EEG Proce...,Neurology
4867696,1487652053,1487652053,95800,2018-03-01,19,28,-1960.63,NaN,0,Sleep Medicine Testing and Long-term EEG Proce...,1,Sleep Medicine Testing and Long-term EEG Proce...,Neurology


In [11]:
monthly = (
    combined
    .groupby(["HCPCS_CODE", "CLAIM_FROM_MONTH"], as_index=False)
    .agg({
        "CATEGORY_DESCRIPTION": "first",
        "SPECIALTY": "first",
        "TOTAL_PAID": "sum",
        "TOTAL_CLAIMS": "sum",
        "TOTAL_UNIQUE_BENEFICIARIES": "sum"
    })
    .sort_values(["HCPCS_CODE", "CLAIM_FROM_MONTH"])
)

monthly

,HCPCS_CODE,CLAIM_FROM_MONTH,CATEGORY_DESCRIPTION,SPECIALTY,TOTAL_PAID,TOTAL_CLAIMS,TOTAL_UNIQUE_BENEFICIARIES
0,92920,2023-03-01,Therapeutic Cardiovascular Services and Proced...,Cardiovascular,0.00,14,14
1,92928,2018-01-01,Therapeutic Cardiovascular Services and Proced...,Cardiovascular,35255.22,78,59
2,92928,2018-02-01,Therapeutic Cardiovascular Services and Proced...,Cardiovascular,37203.82,86,63
3,92928,2018-03-01,Therapeutic Cardiovascular Services and Proced...,Cardiovascular,32756.61,96,81
4,92928,2018-04-01,Therapeutic Cardiovascular Services and Proced...,Cardiovascular,35420.93,108,81
...,...,...,...,...,...,...,...
12096,C2629,2024-07-01,Assorted Cardiovascular and Genitourinary Devices,Cardiovascular,1685.87,35,30
12097,C2629,2024-08-01,Assorted Cardiovascular and Genitourinary Devices,Cardiovascular,1385.10,47,41
12098,C2629,2024-09-01,Assorted Cardiovascular and Genitourinary Devices,Cardiovascular,510.30,20,16
12099,C2629,2024-10-01,Assorted Cardiovascular and Genitourinary Devices,Cardiovascular,291.60,13,13


In [12]:
code_lookup = (
    monthly[["HCPCS_CODE", "CATEGORY_DESCRIPTION", "SPECIALTY"]]
    .drop_duplicates()
    .sort_values(["SPECIALTY", "HCPCS_CODE"])
)

code_lookup.head()

,HCPCS_CODE,CATEGORY_DESCRIPTION,SPECIALTY
0,92920,Therapeutic Cardiovascular Services and Proced...,Cardiovascular
1,92928,Therapeutic Cardiovascular Services and Proced...,Cardiovascular
82,92941,Therapeutic Cardiovascular Services and Proced...,Cardiovascular
83,92950,Therapeutic Cardiovascular Services and Proced...,Cardiovascular
157,92953,Therapeutic Cardiovascular Services and Proced...,Cardiovascular


In [13]:
claims_pivot = (
    monthly
    .pivot(index="CLAIM_FROM_MONTH", columns="HCPCS_CODE", values="TOTAL_CLAIMS")
    .sort_index()
)

claims_pivot

HCPCS_CODE,92920,92928,92941,92950,92953,92960,92971,92973,92975,92978,...,95941,C2617,C2618,C2623,C2624,C2625,C2626,C2627,C2628,C2629
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,78.0,12.0,48.0,NaN,NaN,758.0,NaN,NaN,NaN,...,191.0,343.0,NaN,NaN,NaN,55.0,NaN,93.0,NaN,21.0
2018-02-01,NaN,86.0,NaN,29.0,NaN,NaN,553.0,NaN,NaN,NaN,...,158.0,218.0,NaN,14.0,NaN,18.0,NaN,42.0,NaN,18.0
2018-03-01,NaN,96.0,NaN,61.0,NaN,NaN,669.0,NaN,NaN,NaN,...,236.0,292.0,NaN,19.0,NaN,NaN,NaN,65.0,12.0,NaN
2018-04-01,NaN,108.0,NaN,20.0,NaN,NaN,535.0,NaN,NaN,NaN,...,229.0,361.0,NaN,NaN,NaN,NaN,NaN,78.0,NaN,NaN
2018-05-01,NaN,188.0,NaN,37.0,NaN,15.0,551.0,NaN,NaN,NaN,...,135.0,342.0,NaN,NaN,NaN,13.0,NaN,55.0,15.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,64.0,NaN,NaN,NaN,NaN,114.0,NaN,NaN,74.0,...,584.0,526.0,32.0,NaN,NaN,71.0,NaN,34.0,28.0,47.0
2024-09-01,NaN,71.0,NaN,12.0,NaN,NaN,101.0,NaN,NaN,28.0,...,426.0,348.0,NaN,NaN,NaN,53.0,NaN,34.0,13.0,20.0
2024-10-01,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.0,...,506.0,369.0,13.0,NaN,NaN,105.0,NaN,55.0,NaN,13.0


In [ ]:
claims_pivot.write_csv("claims_pivot.csv")

In [14]:
full_months = pd.date_range(
    monthly["CLAIM_FROM_MONTH"].min(),
    monthly["CLAIM_FROM_MONTH"].max(),
    freq="MS"
)
claims_pivot = claims_pivot.reindex(full_months)
claims_pivot.index.name = "CLAIM_FROM_MONTH"
claims_pivot

HCPCS_CODE,92920,92928,92941,92950,92953,92960,92971,92973,92975,92978,...,95941,C2617,C2618,C2623,C2624,C2625,C2626,C2627,C2628,C2629
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,78.0,12.0,48.0,NaN,NaN,758.0,NaN,NaN,NaN,...,191.0,343.0,NaN,NaN,NaN,55.0,NaN,93.0,NaN,21.0
2018-02-01,NaN,86.0,NaN,29.0,NaN,NaN,553.0,NaN,NaN,NaN,...,158.0,218.0,NaN,14.0,NaN,18.0,NaN,42.0,NaN,18.0
2018-03-01,NaN,96.0,NaN,61.0,NaN,NaN,669.0,NaN,NaN,NaN,...,236.0,292.0,NaN,19.0,NaN,NaN,NaN,65.0,12.0,NaN
2018-04-01,NaN,108.0,NaN,20.0,NaN,NaN,535.0,NaN,NaN,NaN,...,229.0,361.0,NaN,NaN,NaN,NaN,NaN,78.0,NaN,NaN
2018-05-01,NaN,188.0,NaN,37.0,NaN,15.0,551.0,NaN,NaN,NaN,...,135.0,342.0,NaN,NaN,NaN,13.0,NaN,55.0,15.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,64.0,NaN,NaN,NaN,NaN,114.0,NaN,NaN,74.0,...,584.0,526.0,32.0,NaN,NaN,71.0,NaN,34.0,28.0,47.0
2024-09-01,NaN,71.0,NaN,12.0,NaN,NaN,101.0,NaN,NaN,28.0,...,426.0,348.0,NaN,NaN,NaN,53.0,NaN,34.0,13.0,20.0
2024-10-01,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.0,...,506.0,369.0,13.0,NaN,NaN,105.0,NaN,55.0,NaN,13.0


$Y_t​=B_0​+B_1​T+B_2​X_t​+B_3​(X_t​⋅T)$

In [31]:
code = "93247"
intervention_month = pd.Timestamp("2021-01-01")

ts = claims_pivot[[code]].reset_index()
ts.columns = ["month", "y"]

ts["y"] = ts["y"].fillna(0)

# time trend
ts["T"] = np.arange(len(ts))

# intervention dummy
ts["X_t"] = (ts["month"] >= intervention_month).astype(int)

# time since intervention
ts["T'"] = 0
if (ts["month"] >= intervention_month).any():
    intervention_idx = ts.loc[ts["month"] >= intervention_month, "T"].min()
    ts.loc[ts["month"] >= intervention_month, "T'"] = (
        ts.loc[ts["month"] >= intervention_month, "T"] - intervention_idx + 1
    )

ts.head(50)

,month,y,T,X_t,T'
0,2018-01-01,0.0,0,0,0
1,2018-02-01,0.0,1,0,0
2,2018-03-01,0.0,2,0,0
3,2018-04-01,0.0,3,0,0
4,2018-05-01,0.0,4,0,0
5,2018-06-01,0.0,5,0,0
6,2018-07-01,0.0,6,0,0
7,2018-08-01,0.0,7,0,0
8,2018-09-01,0.0,8,0,0
9,2018-10-01,0.0,9,0,0


In [32]:
X = sm.add_constant(ts[["T", "X_t", "T'"]])
y = ts["y"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.903
Model:                            OLS   Adj. R-squared:                  0.899
Method:                 Least Squares   F-statistic:                     247.0
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           2.44e-40
Time:                        22:08:22   Log-Likelihood:                -671.85
No. Observations:                  84   AIC:                             1352.
Df Residuals:                      80   BIC:                             1361.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.293e-12    240.897   5.37e-15      1.0

In [27]:
X

,const,T,X_t,T'
0,1.0,0,0,0
1,1.0,1,0,0
2,1.0,2,0,0
3,1.0,3,0,0
4,1.0,4,0,0
...,...,...,...,...
79,1.0,79,1,44
80,1.0,80,1,45
81,1.0,81,1,46
82,1.0,82,1,47


In [28]:
def fit_its_single(ts, intervention_month):
    ts = ts.copy()

    # outcome
    ts["y"] = ts["y"].fillna(0)

    # time
    ts["T"] = np.arange(len(ts))

    # intervention indicator
    ts["X_t"] = (ts["month"] >= intervention_month).astype(int)

    # post time
    ts["post_time"] = 0
    if (ts["month"] >= intervention_month).any():
        intervention_idx = ts.loc[ts["month"] >= intervention_month, "T"].min()
        ts.loc[ts["month"] >= intervention_month, "post_time"] = (
            ts.loc[ts["month"] >= intervention_month, "T"] - intervention_idx + 1
        )

    # design matrix
    X = sm.add_constant(ts[["T", "X_t", "post_time"]])
    y = ts["y"]

    # fit model
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 1})

    return model

In [29]:
results = []

intervention_month = pd.Timestamp("2021-01-01")

for code in claims_pivot.columns:

    # build time series
    ts = claims_pivot[[code]].reset_index()
    ts.columns = ["month", "y"]

    # skip sparse series
    observed = claims_pivot[code].notna().sum()
    if observed < 8:
        continue

    # fit model
    try:
        model = fit_its_single(ts, intervention_month)

        params = model.params
        pvals = model.pvalues
        conf = model.conf_int()

        # lookup description
        info = code_lookup.loc[
            code_lookup["HCPCS_CODE"].astype(str) == str(code)
        ]

        desc = info["CATEGORY_DESCRIPTION"].iloc[0] if len(info) else None

        results.append({
            "HCPCS_CODE": code,
            "DESCRIPTION": desc,

            # coefficients
            "B0_const": params.get("const", np.nan),
            "B1_time": params.get("T", np.nan),
            "B2_level": params.get("X_t", np.nan),
            "B3_slope": params.get("post_time", np.nan),

            # p-values
            "p_const": pvals.get("const", np.nan),
            "p_time": pvals.get("T", np.nan),
            "p_level": pvals.get("X_t", np.nan),
            "p_slope": pvals.get("post_time", np.nan),

            # confidence intervals
            "ci_const_low": conf.loc["const", 0] if "const" in conf.index else np.nan,
            "ci_const_high": conf.loc["const", 1] if "const" in conf.index else np.nan,

            "ci_time_low": conf.loc["T", 0] if "T" in conf.index else np.nan,
            "ci_time_high": conf.loc["T", 1] if "T" in conf.index else np.nan,

            "ci_level_low": conf.loc["X_t", 0] if "X_t" in conf.index else np.nan,
            "ci_level_high": conf.loc["X_t", 1] if "X_t" in conf.index else np.nan,

            "ci_slope_low": conf.loc["post_time", 0] if "post_time" in conf.index else np.nan,
            "ci_slope_high": conf.loc["post_time", 1] if "post_time" in conf.index else np.nan,

            # model fit
            "R_squared": model.rsquared,
            "N_obs": int(model.nobs)
        })

    except Exception as e:
        results.append({
            "HCPCS_CODE": code,
            "DESCRIPTION": None,
            "ERROR": str(e)
        })

In [30]:
results_df = pd.DataFrame(results)

# optional sorting
results_df = results_df.sort_values("p_level")

results_df

,HCPCS_CODE,DESCRIPTION,B0_const,B1_time,B2_level,B3_slope,p_const,p_time,p_level,p_slope,ci_const_low,ci_const_high,ci_time_low,ci_time_high,ci_level_low,ci_level_high,ci_slope_low,ci_slope_high,R_squared,N_obs
33,93247,Cardiovascular Monitoring Services,1.293281e-12,2.443157e-14,3127.538121,48.211029,NaN,NaN,1.423691e-25,3.879422e-03,NaN,NaN,NaN,NaN,2541.101574,3713.974667,15.490007,80.932051,0.902548,84
29,93243,Cardiovascular Monitoring Services,1.150573e-12,2.352196e-14,2688.827128,66.160117,1.000000e+00,1.000000e+00,3.938932e-16,1.655215e-04,-0.000053,0.000053,-0.000004,0.000004,2041.440238,3336.214018,31.732967,100.587267,0.894567,84
32,93246,Cardiovascular Monitoring Services,2.513847e-13,8.718814e-15,626.835106,28.205710,NaN,NaN,1.517405e-07,4.622065e-05,NaN,NaN,NaN,NaN,392.840352,860.829861,14.636065,41.775355,0.847137,84
34,93248,Cardiovascular Monitoring Services,2.711560e-13,8.916539e-15,655.897163,32.252497,NaN,NaN,1.588574e-07,8.687559e-06,NaN,NaN,NaN,NaN,410.659042,901.135284,18.039178,46.465816,0.850891,84
157,95868,Electromyography Procedures,6.430931e+00,-1.563707e-01,22.577503,-0.300859,4.179156e-02,2.127229e-01,3.910418e-07,7.397418e-02,0.238921,12.622941,-0.402320,0.089579,13.855099,31.299907,-0.630880,0.029162,0.231493,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,C2628,Assorted Cardiovascular and Genitourinary Devices,8.376877e+00,6.576577e-02,-0.945523,0.069275,4.188129e-04,6.111345e-01,8.134393e-01,7.044814e-01,3.723058,13.030696,-0.187745,0.319276,-8.798309,6.907263,-0.288714,0.427265,0.035259,84
68,93316,Echocardiography Procedures,3.525526e+00,1.287001e-01,-0.639073,-0.297719,1.186256e-01,2.298831e-01,8.608586e-01,2.402493e-02,-0.902286,7.953337,-0.081391,0.338791,-7.785067,6.506922,-0.556287,-0.039151,0.090584,84
129,95723,Sleep Medicine Testing and Long-term EEG Proce...,-4.677177e+00,5.259974e-01,-0.837343,-0.387374,3.392121e-02,4.680396e-03,8.964514e-01,1.457332e-01,-8.999227,-0.355127,0.161483,0.890512,-13.447563,11.772878,-0.909270,0.134523,0.258242,84
147,95831,Electrocorticography,4.565030e+02,-1.301763e+01,-0.885886,13.017632,7.341956e-45,1.255380e-17,9.818460e-01,1.255380e-17,392.836612,520.169394,-16.002530,-10.032734,-77.191592,75.419820,10.032734,16.002530,0.803691,84
